# Directors

## Imports

In [1]:
import os
import itertools

import cv2
import duckdb
import requests
import numpy as np
import pandas as pd
import sqlalchemy as db
import plotly.express as px
import plotly.graph_objects as go
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial import distance
from huggingface_hub import hf_hub_download
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm 
from sklearn.linear_model import LinearRegression

/home/amos/anaconda3/envs/face/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if os.path.exists('utils.py'):
    os.remove('utils.py')

raw_url = "https://raw.githubusercontent.com/astaileyyoung/CineFace/research/research/utils.py"

response = requests.get(raw_url)

if response.status_code == 200:
    with open("utils.py", "wb") as f:
        f.write(response.content)
    import utils
    print("✅ Success! utils.py is now actual code.")
else:
    print(f"❌ Failed to download. Error code: {response.status_code}")

❌ Failed to download. Error code: 404


## Setup

## Load Data

In [14]:
# dw_local_path = hf_hub_download(
#     repo_id="astaileyyoung/CineFaceDB",
#     filename="CineFaceDW.duckdb",
#     repo_type="dataset",
#     local_dir="."
# )

In [ ]:
# conn = duckdb.connect("/home/amos/datasets/CineFace/CineFaceDW.duckdb")

In [17]:
username = "amos"
password = "M0$hicat"
host = "192.168.0.131"
port = "3306"
database = "CineFaceDW"
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [ ]:
# df = pd.read_sql_query("SELECT * FROM vwWorksByDirector WHERE kind = 'movie'", conn)
# df

,person_id,person_name,title,year,kind,runtime,fact_id,imdb_id,work_id,avg_size,...,z_v_f_per_fr,z_v_f_per_fr_g,z_vert,z_vert_g,z_v_size,z_v_size_g,z_gini,z_gini_g,z_h_spread,z_h_spread_g
0,100036,D.W. Griffith,The Avenging Conscience: or 'Thou Shalt Not Kill',1914,movie,78.0,254,3643,32807,0.0176,...,-0.803757,-0.877772,-0.459617,0.438055,1.798280,-0.338659,-0.855503,-0.508431,0.889658,0.144955
1,41611,Giovanni Pastrone,Cabiria,1914,movie,148.0,259,3740,32809,0.0056,...,-0.538534,-0.639994,-0.444883,0.453075,-0.504543,-1.250750,-0.193355,0.289892,0.203773,-0.620781
2,1037794,Lois Weber,Hypocrites,1915,movie,54.0,273,4134,32812,0.0090,...,0.659936,0.457404,-1.505170,-0.913772,0.438630,-0.893515,-1.452290,-1.359710,0.503574,-0.360852
3,100036,D.W. Griffith,Judith of Bethulia,1914,movie,61.0,275,4181,32813,0.0087,...,-0.879711,-0.945867,0.547204,1.464440,0.157520,-0.988525,0.687312,1.351680,-1.853880,-2.917990
4,1307858,Colin Campbell,The Spoilers,1914,movie,110.0,292,4630,32816,0.0101,...,0.066493,-0.097575,-0.204230,0.698405,-0.159119,-1.113940,0.036145,0.566592,0.449182,-0.346802
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7336,190,Clint Eastwood,Richard Jewell,2019,movie,131.0,9605,3513548,56220,0.0254,...,0.722229,0.937023,-0.693659,-0.703489,-0.201171,-0.175242,-1.393320,-1.293520,-0.180232,0.436496
7337,525,Christopher Nolan,Dunkirk,2017,movie,106.0,9647,5013056,56222,0.0274,...,1.042170,1.674370,0.622104,0.252804,0.057086,0.258002,0.542099,0.060854,-1.685640,-0.497842
7338,6409,Robert Salerno,Here After,2024,movie,93.0,9655,5402714,56223,0.0472,...,0.047633,0.081220,-1.022900,-0.808631,-0.164107,0.592436,-0.312975,-0.092720,0.754561,0.555923
7339,7467,David Fincher,Mank,2020,movie,131.0,9363,10618286,56225,0.0193,...,-0.472094,-0.196085,-0.120788,-0.312962,-0.704004,-0.490674,-0.329387,-0.610373,-1.632300,-0.487304


In [ ]:
# g = df.groupby('person_id')[[
#     'z_size', 
#     'z_size_g',
#     'z_v_size', 
#     'z_v_size_g',
#     'person_name', 
#     'person_id',
#     'pct_mc',
#     'z_f_per_fr',
#     'z_f_per_fr_g',
#     'z_v_f_per_fr',
#     'z_v_f_per_fr_g',
#     'z_gini',
#     "z_gini_g",
#     'z_dist',
#     'z_dist_g',
#     'z_disp',
#     'z_disp_g',
#     'z_v_dist',
#     'z_v_dist_g',
#     'z_vert',
#     'z_vert_g',
#     'z_h_spread',
#     'z_h_spread_g'

#     ]].agg(
#     {
#         "z_size": ["min", "mean", "max"],
#         "z_size_g": "mean",
#         "z_v_size": ["min", "mean", "max"],
#         "z_v_size_g": "mean",
#         "pct_mc": "mean",
#         "z_f_per_fr": "mean",
#         "z_f_per_fr_g": "mean",
#         "z_v_f_per_fr": "mean",
#         "z_v_f_per_fr_g": "mean",
#         "z_gini": "mean",
#         "z_gini_g": "mean",
#         "z_dist": "mean",
#         "z_dist_g": "mean",
#         "z_disp": "mean",
#         "z_disp_g": "mean",
#         "z_v_dist": "mean",
#         "z_v_dist_g": "mean",
#         "z_vert": "mean",
#         "z_vert_g": "mean",
#         "z_h_spread": "mean",
#         "z_h_spread_g": "mean",
#         "person_name": "max",
#         "person_id": "count"
#     }
# ).rename(
#     {
#         "person_id": "cnt"
#     }, axis=1
# )
# g = g[g['cnt']['count'] > 5]
# g.columns = ['_'.join(col).strip() for col in g.columns.values]
# g = g.reset_index()
# g['z_size_range'] = g['z_size_max'] - g['z_size_min']
# g['z_v_size_range'] = g['z_v_size_max'] - g['z_v_size_min']
# g = g.rename({"person_name_max": "name"}, axis=1)
# g

,person_id,z_size_min,z_size_mean,z_size_max,z_size_g_mean,z_v_size_min,z_v_size_mean,z_v_size_max,z_v_size_g_mean,pct_mc_mean,...,z_v_dist_mean,z_v_dist_g_mean,z_vert_mean,z_vert_g_mean,z_h_spread_mean,z_h_spread_g_mean,name,cnt_count,z_size_range,z_v_size_range
0,40,-0.662475,0.682375,2.803430,-0.104401,0.276332,1.000943,2.362290,0.129211,0.386267,...,1.355065,-0.086482,-0.049015,0.005247,-1.061491,-0.899443,Orson Welles,9,3.465905,2.085958
1,68,-1.800720,-0.451103,1.557080,-0.688960,-1.278470,-0.057186,1.799650,-0.529139,0.189997,...,0.450576,-0.546360,-0.178344,0.135372,-0.057919,-0.189269,Fritz Lang,33,3.357800,3.078120
2,138,-2.108400,0.157275,3.002120,0.337829,-1.547210,0.699764,3.100580,0.743365,0.298943,...,-0.719243,-0.144341,0.920641,0.506002,-0.553557,-0.274543,Quentin Tarantino,7,5.110520,4.647790
3,190,-1.250860,-0.165057,1.281890,0.083618,-1.026280,-0.101402,1.197400,0.171068,0.301569,...,-2.663803,-1.214265,-0.471236,-0.596468,0.211499,0.237269,Clint Eastwood,32,2.532750,2.223680
4,240,-1.378070,-0.118404,1.488330,-0.288452,-1.254410,0.389179,2.682740,0.168059,0.327817,...,1.626615,0.664690,0.860518,0.695902,-0.367702,-0.275966,Stanley Kubrick,12,2.866400,3.937150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308,1021685,-1.346220,-0.415989,1.290740,-0.611928,-1.224860,-0.325608,1.225870,-0.601519,0.184417,...,-0.328302,-0.943810,0.813044,1.231625,-0.364781,-0.795822,E. Mason Hopper,6,2.636960,2.450730
309,1036754,-0.193419,0.766000,1.892600,-0.631529,0.099949,0.754892,2.757180,-0.626946,0.168900,...,0.393365,-0.855220,-0.251113,0.475963,0.012690,-0.608738,Reginald Barker,7,2.086019,2.657231
310,1065336,-1.263120,-0.757020,-0.216518,-0.796533,-1.110990,-0.760721,-0.214824,-0.846643,0.158433,...,-0.440322,-0.953394,0.501104,0.856954,0.157485,-0.175273,Melville W. Brown,6,1.046602,0.896166
311,1188853,-1.111600,-0.836762,-0.677957,-1.140797,-1.388340,-0.785310,-0.293239,-1.109503,0.254517,...,0.020088,-0.984792,-0.176227,0.690061,-0.307814,-1.253625,Georg af Klercker,6,0.433643,1.095101


In [20]:
g = pd.read_sql_query("SELECT * FROM vwDirectorStats WHERE cnt > 5;", conn)
g

,person_id,name,cnt,z_size_mean,z_size_g_mean,z_size_std,z_size_g_std,z_v_size_mean,z_v_size_g_mean,z_v_size_std,...,z_v_dist_g_std,z_v_dist_std,z_vert_mean,z_vert_g_mean,z_vert_g_std,z_vert_std,z_h_spread_mean,z_h_spread_g_mean,z_h_spread_g_std,z_h_spread_std
0,100036,D.W. Griffith,20,0.847967,-0.617665,0.849931,0.263196,0.834501,-0.553381,0.819831,...,0.198507,0.664295,-0.106748,0.811058,0.601138,0.713173,-0.089726,-1.024900,1.094353,1.030350
1,8636,Cecil B. DeMille,19,0.104122,-0.730753,0.870612,0.279380,-0.265491,-0.806106,0.666110,...,0.416896,0.760424,-1.323008,-0.486616,0.759664,0.849092,0.486764,-0.233846,0.782604,0.649938
2,72061,Frank Lloyd,19,-0.066855,-0.607070,0.513403,0.196986,0.010675,-0.578883,0.789236,...,0.270958,0.972972,0.151392,0.613291,0.747290,0.749806,0.165656,-0.216838,0.628500,0.565331
3,90375,Christy Cabanne,18,0.563096,-0.385190,0.541358,0.220025,0.217626,-0.517277,0.563077,...,0.071411,0.397960,0.279091,0.655014,0.743446,0.883555,0.088884,-0.060334,1.005605,1.067817
4,42060,Sidney Franklin,10,0.924703,-0.471228,1.087197,0.327850,0.597186,-0.535139,1.102163,...,0.161067,0.505395,-0.167981,0.664360,0.731679,0.849490,-0.014050,-0.924616,0.849258,0.886667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308,1188853,Georg af Klercker,6,-0.836763,-1.140798,0.154889,0.084414,-0.785309,-1.109504,0.384313,...,0.032468,0.286596,-0.176228,0.690061,0.636455,0.682395,-0.307813,-1.253625,0.433087,0.449338
309,6817,Agnès Varda,8,-0.523586,-0.341256,0.379753,0.302244,-0.328108,-0.105885,0.597030,...,0.430853,0.890803,-0.430183,-0.476933,0.857787,0.943240,-0.170326,-0.121999,0.859515,0.981758
310,510,Tim Burton,6,-0.133883,0.311100,0.594104,0.493867,0.038107,0.446121,0.619037,...,0.115432,1.224537,-0.549724,-0.801954,0.613859,0.738224,-0.236135,-0.213911,0.373270,0.637907
311,21684,Bong Joon Ho,8,0.015766,0.251852,0.422095,0.356903,0.517931,0.658467,0.587749,...,0.152516,0.944485,-0.167324,-0.393070,0.610322,0.769178,-0.451087,-0.071067,0.174562,0.495652


## Analysis

### Overall

#### Face Size vs. Size Variance

The first measure is relativley simple; it's simply the average size of faces within a film averaged over a director's career. This is plotted against how much the size of the face varies within a film. As we can see, the correlation between these two variables is rather tight.

In [21]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_sample(g, x, y, name_field="name")

What might be an explanation? Well, the larger the average face size, the more 'dynamic range' the director has available. A director who favors long shots (like Chaplin) is mathematically restricted to a narrow band of variance because the face is already small. A director who favors close-ups (like Bergman) has the 'spatial budget' to swing wildly between different scales. Therefore, shot-size variety is a luxury of the close-up.

#### Face Size vs. Career Volatility

This measures the average scale of the face within a film against a director's tendency to vary that scale *betweeen* films. A high score on the x axis means that the director highly privileges close-ups while a low score means they favor long shots. A high score on the y axis means that a director changes this tendency between films while a low score means that the director remains largely consistent in their shot scales across their career. 

In [22]:
x = "z_size_g_mean"
y = "z_size_g_std"
plot_sample(g, x, y, name_field="name")

In [23]:
g[["name", "z_size_g_mean"]].sort_values('z_size_g_mean')[:10]

,name,z_size_g_mean
308,Georg af Klercker,-1.140798
35,Buster Keaton,-1.101715
171,Kenji Mizoguchi,-1.001639
19,Charlie Chaplin,-0.981970
204,Fred C. Brannon,-0.913783
158,Howard Bretherton,-0.876938
10,Victor Sjöström,-0.862892
135,Ray Taylor,-0.851224
16,Fred Niblo,-0.841794
200,Hal Walker,-0.838943


In [24]:
g[['name', 'z_size_g_mean']].sort_values('z_size_g_mean', ascending=False)[:10]

,name,z_size_g_mean
298,Tony Scott,1.835344
300,Wong Kar-Wai,1.587374
303,Christopher Nolan,1.369838
307,Paul Thomas Anderson,1.103158
287,David Lynch,1.091378
248,Ingmar Bergman,1.037406
305,Denis Villeneuve,0.947241
299,Ridley Scott,0.945289
294,Michael Mann,0.881964
297,Roger Donaldson,0.865986


#### Face Variance vs. Career Volatility

This plot represents how the variance in face size within a film contrasts with how consistently the face size varies *between* movies. Ingmar Bergman, located in the top right corner, not only varies his face sizes within a film, but between his films. 

In [25]:
x = "z_v_size_g_mean"   # How much do face sizes vary within a movie
y = "z_v_size_g_std"    # How much does a director vary his/her shot scale from one film to the next
plot_sample(g, x, y, name_field="name")

#### Face Size Variance vs. Face Variance Variance

In [26]:
"""
On the x axis is the STDDEV of average face size. How much does a director vary their face sizes between films. 
If a director always uses the same shot scale across their career--close-ups, for instance--they will have a low value on the x axis.

On the y axis is the STDDEV of face size variance. How much do the face sizes vary within a film, and how much does this change between films.
If a director varies their scales a lot within a movie, how much do they do this across their career.
"""
x = 'z_size_g_std'
y = "z_v_size_g_std"    
plot_sample(g, x, y)

In [27]:
g[["name", "z_v_size_g_std"]].sort_values(by="z_v_size_g_std")[:10]

,name,z_v_size_g_std
140,Ralph Staub,0.089253
35,Buster Keaton,0.097263
127,Bernard B. Ray,0.104724
308,Georg af Klercker,0.108769
204,Fred C. Brannon,0.112153
145,Philip Ford,0.119771
90,Louis King,0.122588
57,Howard Higgin,0.126369
158,Howard Bretherton,0.126711
38,Albert S. Rogell,0.127056


In [28]:
g[["name", "z_v_size_g_std"]].sort_values(by="z_v_size_g_std", ascending=False)[:10]

,name,z_v_size_g_std
271,Sergio Leone,1.185636
248,Ingmar Bergman,1.120860
287,David Lynch,1.064515
304,Quentin Tarantino,0.991909
186,Emeric Pressburger,0.889803
312,Yorgos Lanthimos,0.869083
300,Wong Kar-Wai,0.855396
224,Vittorio De Sica,0.801854
242,Robert Parrish,0.800443
305,Denis Villeneuve,0.775416


In [29]:
X = g['z_size_g_std'].values.reshape(-1, 1)
y = g['z_v_size_g_std'].values
model = LinearRegression().fit(X, y)
# 2. Calculate the 'Expected' variance for each director
g['expected_v_g_std'] = model.predict(X)
g['style_pivot_g'] = g['z_v_size_g_std'] - g['expected_v_g_std']

X = g['z_size_std'].values.reshape(-1, 1)
y = g['z_v_size_std'].values
model = LinearRegression().fit(X, y)
# 2. Calculate the 'Expected' variance for each director
g['expected_v_std'] = model.predict(X)
g['style_pivot'] = g['z_v_size_g_std'] - g['expected_v_std']

In [30]:
g.sort_values(by='style_pivot', ascending=True)[["name", "style_pivot"]]

,name,style_pivot
77,Norman Z. McLeod,-1.240463
213,Delmer Daves,-0.971718
5,Chester M. Franklin,-0.903843
58,Robert Florey,-0.888026
205,Phil Karlson,-0.840428
...,...,...
271,Sergio Leone,0.036321
312,Yorgos Lanthimos,0.078082
283,Brian De Palma,0.080291
224,Vittorio De Sica,0.124041


#### Face Size vs. Avg. Distance

As you can see below, face size and distance to center of the frame are slightly inversely correlated. This makes sense intuitively because the larger a face is, the less room there is to play within the frame. 

In [31]:
x = "z_size_g_mean"
y = "z_dist_g_mean"
plot_sample(g, x, y)

#### Face Size Variance vs. Horizontal Spread Variance

In [32]:
x = "z_v_size_g_mean"
y = "z_h_spread_g_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Face Size

This result was a little surprising to me. Yes, there is an inverse relationship between the number of faces per frame and the size of the face. However, it's weaker than I expected. My guess would be that this is because we're aggregating at the film level instead of the frame level. I imagine if we aggregated by frame we might see a different result. There are two significant outliers: Stanley Kramer and Ingmar Bergman. Kramer puts a lot of faces into the frame, but maintains an average face size. This means that the frame is packed with faces. On the other end, Bergman is in the bottom 7 of faces per frame, but is an outlier on face size, as we've already seen.

In [33]:
x = "z_f_per_fr_mean"
y = "z_size_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Face Size Variance

Because average face size and avg face variance are highly correlated, the relationship between variance and faces per frame is essentially the same plot.

In [34]:
x = "z_f_per_fr_mean"
y = "z_v_size_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Avg Distance from Center

There is a mild positive relationship between faces per frame and the average distance to the center of the frame. 

In [35]:
x = "z_f_per_fr_mean"
y = "z_dist_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Variance of Distance to Center

In [36]:
x = "z_f_per_fr_mean"
y = "z_v_dist_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Dispersion

In [37]:
x = "z_f_per_fr_g_mean"
y = "z_disp_g_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Gini

In [38]:
x = "z_f_per_fr_g_mean"
y = "z_gini_g_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Variance of Faces per Frame

In [39]:
x = "z_f_per_fr_g_mean"
y = "z_v_f_per_fr_g_mean"
plot_sample(g, x, y)

#### Distance from Center vs. Gini

In [40]:
x = "z_dist_g_mean"
y = "z_gini_g_mean"
plot_sample(g, x, y)

#### Distance from Center vs. Dispersion

In [41]:
x = "z_dist_g_mean"
y = "z_disp_g_mean"
plot_sample(g, x, y)

#### Distance from Center Average vs. Distance from Center Variance

Somewhat surprising result: the average distance of a face from the center of the image is completely unrelated to how 

In [42]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_sample(g, x, y)

#### Distance from Center Film Variance vs. Distance from Center Career Variance

In [43]:
x = "z_dist_g_std"
y = "z_v_dist_g_std"
plot_sample(g, x, y)

#### Distance from Center Variance vs. Gini

In [44]:
x = "z_v_dist_g_mean"
y = "z_gini_g_mean"
plot_sample(g, x, y)

#### Distance from Center Variance vs. Dispersion

In [45]:
x = "z_v_dist_g_mean"
y = "z_disp_g_mean"
plot_sample(g, x, y)

#### Distance from Center Variance vs. Vertical Discipline

In [46]:
x = "z_v_dist_g_mean"
y = "z_vert_g_mean"
plot_sample(g, x, y)

#### Distance to Center vs. Horizontal Spread

In [47]:
x = "z_v_dist_g_mean"
y = "z_h_spread_g_mean"
plot_sample(g, x, y, name_field="person_name_max")

ValueError: Value of 'hover_name' is not the name of a column in 'data_frame'. Expected one of ['person_id', 'name', 'cnt', 'z_size_mean', 'z_size_g_mean', 'z_size_std', 'z_size_g_std', 'z_v_size_mean', 'z_v_size_g_mean', 'z_v_size_std', 'z_v_size_g_std', 'pct_mc_mean', 'pct_mc_std', 'z_f_per_fr_mean', 'z_f_per_fr_g_mean', 'z_f_per_fr_g_std', 'z_f_per_fr_std', 'z_v_f_per_fr_mean', 'z_v_f_per_fr_g_mean', 'z_v_f_per_fr_g_std', 'z_v_f_per_fr_std', 'z_gini_mean', 'z_gini_g_mean', 'z_gini_g_std', 'z_gini_std', 'z_dist_mean', 'z_dist_g_mean', 'z_dist_g_std', 'z_dist_std', 'z_disp_mean', 'z_disp_g_mean', 'z_disp_g_std', 'z_disp_std', 'z_v_dist_mean', 'z_v_dist_g_mean', 'z_v_dist_g_std', 'z_v_dist_std', 'z_vert_mean', 'z_vert_g_mean', 'z_vert_g_std', 'z_vert_std', 'z_h_spread_mean', 'z_h_spread_g_mean', 'z_h_spread_g_std', 'z_h_spread_std', 'expected_v_g_std', 'style_pivot_g', 'expected_v_std', 'style_pivot'] but received: person_name_max

#### Vertical Discipline vs. Dispersion

In [ ]:
x = "z_vert_g_mean"
y = "z_disp_g_mean"
plot_sample(g, x, y)

#### Vertical Discipline vs. Gini

In [ ]:
x = "z_vert_g_mean"
y = "z_gini_g_mean"
plot_sample(g, x, y)

#### Gini vs. Dispersion

In [ ]:
x = "z_gini_g_mean"
y = "z_disp_g_mean"
plot_sample(g, x, y)

In [ ]:
g[["name", "z_disp_g_mean"]].sort_values("z_disp_g_mean")[:10]

,name,z_disp_g_mean
230,Yasujirō Ozu,-1.306215
303,Christopher Nolan,-1.140923
306,Ethan Coen,-1.063507
35,Buster Keaton,-1.028861
15,Alan Crosland,-1.001682
4,Sidney Franklin,-0.955268
135,Ray Taylor,-0.931724
45,E. Mason Hopper,-0.843087
204,Fred C. Brannon,-0.811428
65,George Archainbaud,-0.788838


In [ ]:
g[["name", "z_disp_g_mean"]].sort_values("z_disp_g_mean", ascending=False)[:10]

,name,z_disp_g_mean
267,Stanley Kramer,1.444140
210,Robert Rossen,1.349760
289,Steven Spielberg,1.199059
256,Federico Fellini,1.122348
291,Martin Scorsese,0.908579
283,Brian De Palma,0.850893
262,Sidney Lumet,0.794139
271,Sergio Leone,0.788257
160,Herbert Wilcox,0.778700
220,Richard Fleischer,0.775537


#### Gini vs. Horizontal Spread

In [ ]:
x = "z_gini_g_mean"
y = "z_h_spread_mean"
plot_sample(g, x, y, name_field="person_name_max")

### By Year

#### Face Size vs. Size Variance

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_sample(g, x, y)

#### Face Size vs. Avg. Distance

In [ ]:
x = "z_size_mean"
y = "z_dist_mean"
plot_sample(g, x, y)

#### Face Size vs. Variance Distance

In [ ]:
x = "z_size_mean"
y = "z_dist_mean"
plot_sample(g, x, y)

#### Faces per Frame vs. Face Size

In [ ]:
x = "z_f_per_fr_mean"
y = "z_size_mean"
plot_sample(g, x, y)

#### Face Size vs. Gini Score

In [ ]:
x = "z_size_mean"
y = "z_gini_mean"
plot_sample(g, x, y)

#### Face Size Variance vs. Gini

In [ ]:
x = "z_dist_g_mean"
y = "z_vert_g_mean"
plot_sample(g, x, y)

### Directors

#### Martin Scorsese -- Master of Ensemble Framing

Scorsese presents an interesting case. On a couple metrics, Scorsese is not only not exceptional, but is almost completely average. This is particularly evident when we look at the relationship between the average face size and the average variation in face size, as seen below.

In [ ]:
name = "Martin Scorsese"

##### Face Size vs. Face Variance

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_director_against_sample(g, x, y, name)

On both size and variance, Scorsese is almost completely average. Scorsese also remains relatively consistent across his films, suggesting that 

In [ ]:
x = "z_size_std"
y = "z_v_size_std"
plot_director_against_sample(g, x, y, name)

In [ ]:
g['size_percentile'] = g['z_size_std'].rank(pct=True) * 100
temp = g[g['name'] == "Martin Scorsese"]
temp['size_percentile'].values[0]

38.977635782747605

##### Average Distance from Center vs. Variance Distance from Center

In [ ]:
x = "z_dist_mean"
y = "z_v_dist_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
temp = df[df['person_name'] == "Martin Scorsese"]
px.bar(temp, x='year', y='z_gini')

Only one of Scorsese's movies has a positive Z-score. 

In [ ]:
x = "year"
y = "z_gini"
temp = df[df['person_name'] == "Martin Scorsese"]
fig = px.scatter(temp, 
                 x=x,
                 y=y,
                 hover_name="person_name",
                 hover_data="title",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
fig.show()

In [ ]:
x = "z_f_per_fr_mean"
y = "z_gini_mean"
plot_director_against_sample(g, x, y, name)
# fig = px.scatter(g, 
#                  x=x,
#                  y=y,
#                  hover_name="name",
#                  color_discrete_sequence=["#79b8b8"],
#                  trendline="ols")
# fig.update_layout(layout)
# fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
# fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
# temp = g[g['name'] == "Martin Scorsese"]
# fig.add_trace(go.Scatter(x=temp[x], 
#                          y=temp[y], 
#                          marker=dict(
#                              color="red", 
#                              size=16, 
#                              line=dict(
#                                  color="white",
#                                  width=2
#                              )
#                              )))
# fig.show()

As we can see, Scorsese is almost perfectly average with regard to both average size and average variance. However, if we examine framing, the position of faces within the image, a different picture emerges. When we look at the average Gini score across Scorsese's filmography, we see that Scorsese is, in fact, an outlier.

In [ ]:
x = "z_avg_face_size_mean"
y = "z_composition_gini_mean"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="person_name_max",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['person_name_max'] == "Martin Scorsese"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['person_id', 'name', 'cnt', 'z_size_mean', 'z_size_g_mean', 'z_size_std', 'z_size_g_std', 'z_v_size_mean', 'z_v_size_g_mean', 'z_v_size_std', 'z_v_size_g_std', 'pct_mc_mean', 'pct_mc_std', 'z_f_per_fr_mean', 'z_f_per_fr_g_mean', 'z_f_per_fr_g_std', 'z_v_f_per_fr_mean', 'z_v_f_per_fr_g_mean', 'z_v_f_per_fr_g_std', 'z_gini_mean', 'z_gini_g_mean', 'z_gini_g_std', 'z_dist_mean', 'z_dist_g_mean', 'z_dist_g_std', 'z_disp_mean', 'z_disp_g_mean', 'z_disp_g_std', 'z_v_dist_mean', 'z_v_dist_g_mean', 'z_v_dist_g_std', 'z_vert_mean', 'z_vert_g_mean', 'z_vert_g_std', 'expected_v_g_std', 'style_pivot_g', 'expected_v_std', 'style_pivot', 'size_percentile'] but received: z_avg_face_size_mean

In [ ]:
g['gini_percentile'] = g['z_composition_gini_mean'].rank(pct=True) * 100
temp = g[g['person_name_max'] == "Martin Scorsese"]
temp['gini_percentile'].values[0]

0.3968253968253968

##### Gridmap

Scorsese is not simply an outlier on gini score--he's at the very bottom. This is an interesting finding because Scorsese is quite average regarding the size and variance of the faces yet an outlier on gini score. Why is this the case? And what does this say about Scorsese's framing? Let's look at a gridmap of Scorsese's framing.

In [ ]:
create_gridmap_from_director("Martin Scorsese", dw_conn, layout=layout)

/tmp/ipykernel_6961/3450768217.py:7: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



Scorsese places the faces predominantly in the middle-top and middle-center. This is consistent with intuition--most faces are framed in the middle of the frame. So why does Scorsese score so low on gini score? Let's take a look at the gridmap of the historical average. 

In [ ]:
plot_sample_grid(dw_conn, layout=layout)

The gridmap of the historical average looks quite similar to Scorsese's, with most faces framed in middle-top or middle-center. So what's going on? We can also compare Scorsese's framing compared to the historical average by taking a difference of the two gripmaps. 

In [ ]:
compare_director_to_sample_grid("Martin Scorsese", dw_conn, layout=layout)

As we can see, Scorsese places fewer faces in the middle-top and middle-center cells than the historical average. In conceptual terms, Scorsese's framing is much more "even" or "dispersed" than the historical average--in fact, more than the sample as a whole. Differencing the historical gridmap from Scorsese's tells us *how* Scorsese's gini score is so low, but it doesn't explain *why*. There is another metric, however, that explains why the gini score is so low.

In [ ]:
x = "avg_faces_per_frame_mean"
y = "z_composition_gini_mean"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="person_name_max",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['person_name_max'] == "Martin Scorsese"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

If we look at the average number of faces in a frame, we see that Scorsese likes to place many more faces in the frame than the historical average. This forms an interesting relationship with the average face size and the face size variance. Those two metrics are right around the mean, which means that Scorsese is not adding more faces at the expense of size, as in he doesn't move the camera farther away when adding more faces but instead "crowds" the frame.

In [ ]:
x = "z_avg_face_size"
y = "avg_faces_per_frame"
temp = df[df['person_name'] == "Martin Scorsese"]
fig = px.scatter(temp, 
                 x=x,
                 y=y,
                 hover_name="person_name",
                 hover_data="title",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
fig.add_trace(go.Scatter(
    x=temp[x], 
    y=[temp[y].mean() for _ in temp.values], 
    mode='lines', 
    marker=dict(color="purple"), 
    line=dict(width=4)))
fig.show()

Scorsese has roughly and equal number of films above the mean as below. His average is being pulled up by a few films, particularly *The Aviator*, *Gangs of New York*, and *Taxi Driver*. It would be interesting to see if these averages change significantly with more films added.

#### Ingmar Bergman -- Master of the Close-Up

With regard to face size and variance of face size, Ingmar Bergman is king. He is an extreme outlier, packing the frame with large faces.

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="name",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['name'] == "Ingmar Bergman"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

Bergman is #1 in both average size and average variance. Bergman has a consistently high score for size and variance, but he also has a greater range between films than most other directors.

In [ ]:
x = "z_avg_face_size_range"
y = "z_variance_face_size_range"
fig = px.scatter(g, 
                 x=x,
                 y=y,
                 hover_name="person_name_max",
                 color_discrete_sequence=["#79b8b8"],
                 trendline="ols")
fig.update_layout(layout)
fig.update_layout(
    xaxis=dict(title=x), 
    yaxis=dict(title=y), 
    title=dict(
        text=f"{x} vs. {y}",
        y=0.95
    )
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_traces(selector=dict(mode="lines"), line=dict(dash="dash", color="#d81275", width=5), name="Trend")
temp = g[g['person_name_max'] == "Ingmar Bergman"]
fig.add_trace(go.Scatter(x=temp[x], 
                         y=temp[y], 
                         marker=dict(
                             color="red", 
                             size=16, 
                             line=dict(
                                 color="white",
                                 width=2
                             )
                             )))
fig.show()

As we can see, Bergman is in the top 5 for range. We can see below to see the average across Bergman's films contained in the sample

In [ ]:
temp = df[df['person_name'] == "Ingmar Bergman"].sort_values(by='z_avg_face_size', ascending=False)
fig = px.bar(temp, x='title', y=['z_avg_face_size', 'z_variance_face_size'], barmode="group")
fig.update_layout(layout)
fig.show()

Despite this wide range, none of his films contained in the sample are below average for either face size or variance. *Autumn Sonata* is his most extreme film, in the top 10 for both size and variance.

#### Orson Welles

In [ ]:
name = "Orson Welles"

##### Face Size vs. Face Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_director_against_sample(g, x, y, name)

##### Average Distance from Center vs. Variance Distance from Center

Orson Welles positions his faces farther from the center of the frame than any other director. 

In [ ]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_director_against_sample(g, x, y, name)

##### Average Distance from Center Global vs. Variance Distance from Center Global

But if we normalize by year, a different story emerges. Not only does Welles de-center his images more than anyone else, he is on an island all by himself. Welles is incredibly unique in classical Hollywood. 

In [ ]:
x = "z_dist_mean"
y = "z_v_dist_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_gini_g_mean"
y = "z_disp_g_mean"
plot_director_against_sample(g, x, y, "Orson Welles")

#### Alfred Hitchcock

In [ ]:
name = "Alfred Hitchcock"

If we compare Hitchcock to the global averages, he is 

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_director_against_sample(g, x, y, name)

Compared to directors of his era, Hitchcock more frequently uses close shots. 

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
g['size_percentile'] = g['z_size_mean'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_percentile'].values[0])

g['size_v_percentile'] = g['z_v_size_mean'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_v_percentile'].values[0])

89.83739837398373
88.6178861788618


In [ ]:
x = "z_size_std"
y = "z_v_size_std"
plot_director_against_sample(g, x, y, name)

In [ ]:
g['size_percentile'] = g['z_size_std'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_percentile'].values[0])

g['size_v_percentile'] = g['z_v_size_mean'].rank(pct=True) * 100
temp = g[g['name'] == name]
print(temp['size_v_percentile'].values[0])

In [ ]:
x = "z_size_g_std"
y = "z_v_size_g_std"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_dist_g_mean"
y = "z_v_dist_g_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_dist_mean"
y = "z_v_dist_mean"
plot_director_against_sample(g, x, y, name)

In [ ]:
x = "z_dist_std"
y = "z_v_dist_std"
plot_director_against_sample(g, x, y, name)

NameError: name 'plot_director_against_sample' is not defined

#### Christopher Nolan

In [ ]:
name = "Christopher Nolan"

##### Size vs. Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_director_against_sample(g, x, y, name)

#### Ozu

In [ ]:
name = "Yasujirō Ozu"

##### Size vs. Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Size vs. Size Variance Normalized by Year

In [ ]:
x = "z_size_mean"
y = "z_v_size_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Distance Variance vs. Gini 

In [ ]:
x = "z_v_dist_g_mean"
y = "z_gini_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Vertical Discipline vs. Gini

In [ ]:
x = "z_vert_g_mean"
y = "z_gini_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Faces per Frame vs. Dispersion

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_disp_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

##### Faces per Frame vs. Distance Variance

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_v_dist_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

In [ ]:
x = "z_gini_g_mean"
y = "z_h_spread_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

In [ ]:
x = "z_disp_g_mean"
y = "z_h_spread_g_mean"
plot_directors_against_sample(g, x, y, [name, "Christopher Nolan"])

In [ ]:
features={"x": "z_f_per_fr_g_mean", "y": "z_v_dist_g_mean", "z": "z_gini_g_mean"}
plot_directors_3d(g, [name, "Christopher Nolan"], features=features)

#### Kenji Mizoguchi

In [ ]:
name = "Kenji Mizoguchi"

##### Size vs. Size Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_size_g_mean"
plot_directors_against_sample(g, x, y, [name])

##### Size vs. Distance Variance

In [ ]:
x = "z_size_g_mean"
y = "z_v_dist_g_mean"
plot_directors_against_sample(g, x, y, [name])

##### Faces per Frame vs. Gini

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_gini_g_mean"
plot_directors_against_sample(g, x, y, [name])

##### Faces per Frame vs. Dispersion

In [ ]:
x = "z_f_per_fr_g_mean"
y = "z_disp_g_mean"
plot_directors_against_sample(g, x, y, [name])